# Lokalisierungs-Check: Vorhersage vs. Ground-Truth auf dem Luftbild

Visuelle Verifikation, ob die vorhergesagten Kronen **an der richtigen Stelle** liegen —
auch dort, wo die IoU unter 0.5 bleibt (die 'partial'-Kronen). Grün = GT, Rot = Vorhersage.

**Muss als `nda` in Jupyter laufen** (Inferenz + Bildzugriff auf Data/Kiel).
Parameter unten anpassen (Gebiet / Modell), dann von oben durchlaufen.


In [ ]:
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES","0")
os.environ.setdefault("HSA_OVERRIDE_GFX_VERSION","10.3.0")
import sys; sys.path.insert(0,'/home/leafline/leafline/3_Model/src')
from pathlib import Path
import numpy as np, torch, rasterio, geopandas as gpd
import matplotlib.pyplot as plt, matplotlib.patches as mpatches
from deeptrees.model.deeptrees_model import DeepTreesModel
from deeptrees.modules.utils import predict_on_tile
from deeptrees.modules.postprocessing import extract_polygons
from dataset import channel_indices_for
from evaluate import iou_polygon
import yaml

# ── Parameter ───────────────────────────────────────────────────────────────
CONFIG     = '/home/leafline/leafline/3_Model/configs/finetune_step1_ndom_spring75.yaml'
CHECKPOINT = '/home/leafline/leafline/3_Model/runs/step1_ndom_spring75/checkpoints/best.pt'
AREA       = 'BotGarten'          # oder 'HoernNord'
RES_SUFFIX = ''                   # '' = 7.5cm | '_native20_spring' | '_native20_summer'
GT_FILE    = 'test/%s_GroundTruth.shp' % AREA
IOU_MATCH  = 0.5

cfg = yaml.safe_load(open(CONFIG))
base = Path(cfg['paths']['base']); pp = cfg['postprocessing']
in_ch = cfg['model']['in_channels']; ci = channel_indices_for(in_ch)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ck = torch.load(CHECKPOINT, map_location='cpu', weights_only=False)
model = DeepTreesModel(in_channels=in_ch, apply_sigmoid=True, lr=cfg['model']['lr'], num_backbones=1)
model.load_state_dict(ck['model_state_dict']); model.eval(); model.to(device)
print('Modell geladen:', Path(CHECKPOINT).parent.parent.name, '| in_channels', in_ch)

In [ ]:
# ── Inferenz + Polygone (getuntes Config-PP) ────────────────────────────────
sp = base/'stacked_6ch'/f'{AREA}{RES_SUFFIX}.tif'
with rasterio.open(sp) as src:
    img = src.read().astype(np.float32); transform = src.transform
rgb = np.clip(img[:3].transpose(1,2,0), 0, 1)              # R G B für die Anzeige
xin = img[ci] if ci is not None else img
with torch.no_grad():
    out = predict_on_tile(model, torch.from_numpy(xin).unsqueeze(0).to(device),
                          patch_size=cfg['data']['patch_size'],
                          local_batch_size=cfg['training']['batch_size'],
                          stride=cfg['data']['patch_size']//2)
mask, outline, dist = out[0,0].cpu().numpy(), out[0,1].cpu().numpy(), out[0,2].cpu().numpy()
preds = extract_polygons(mask, outline, dist, transform=transform, **pp)
gts   = list(gpd.read_file(base/GT_FILE).geometry)
inv = ~transform
print(f'{AREA}{RES_SUFFIX}: {len(preds)} Vorhersagen, {len(gts)} GT-Kronen')

# Greedy-Matching @IoU_MATCH → TP/FP/FN + beste IoU pro GT (für Bucket-Färbung)
matched_gt=set(); matched_pred=set()
for pi,p in enumerate(preds):
    bi,bg=0.0,-1
    for gi,g in enumerate(gts):
        if gi in matched_gt: continue
        if p.bounds[0]>g.bounds[2] or p.bounds[2]<g.bounds[0] or p.bounds[1]>g.bounds[3] or p.bounds[3]<g.bounds[1]: continue
        i=iou_polygon(p,g)
        if i>bi: bi,bg=i,gi
    if bi>=IOU_MATCH: matched_pred.add(pi); matched_gt.add(bg)
best_iou_gt=[max([iou_polygon(p,g) for p in preds], default=0.0) for g in gts]
tp=len(matched_pred); fp=len(preds)-tp; fn=len(gts)-len(matched_gt)
print(f'TP={tp}  FP={fp}  FN={fn}  (P={tp/max(tp+fp,1):.3f} R={tp/max(tp+fn,1):.3f})')

def px(poly):
    xs,ys=poly.exterior.xy; c,r=inv*(np.array(xs),np.array(ys)); return c,r

## 1. Gesamtes Gebiet — Grün = GT, Rot = Vorhersage

In [ ]:
fig,ax=plt.subplots(figsize=(14,14)); ax.imshow(rgb); ax.axis('off')
for g in gts:
    c,r=px(g); ax.plot(c,r,color='lime',lw=0.7)
for p in preds:
    c,r=px(p); ax.plot(c,r,color='red',lw=0.7)
ax.set_title(f'{AREA}: GT (grün, {len(gts)}) vs. Vorhersage (rot, {len(preds)})')
ax.legend(handles=[mpatches.Patch(color='lime',label='GT'),mpatches.Patch(color='red',label='Pred')],loc='upper right')
plt.tight_layout(); plt.show()

## 2. Nach Treffer-Status — TP grün · FP rot · verpasste GT nach Überlappung eingefärbt

GT-Kronen: **blau** = keine Überlappung (zero) · **orange** = überlappt aber IoU<0.5 (partial).
Sind die orangen Kronen von roten/grünen Vorhersagen überdeckt, ist die *Lokalisierung* ok —
nur Form/Größe verfehlt die 0.5-Schwelle.

In [ ]:
fig,ax=plt.subplots(figsize=(14,14)); ax.imshow(rgb); ax.axis('off')
for pi,p in enumerate(preds):
    c,r=px(p); col='lime' if pi in matched_pred else 'red'
    ax.fill(c,r,alpha=0.35,fc=col,ec=col,lw=0.6)
for gi,g in enumerate(gts):
    if gi in matched_gt: continue
    c,r=px(g); col='orange' if best_iou_gt[gi]>0 else 'blue'
    ax.plot(c,r,color=col,lw=1.1)
ax.set_title(f'{AREA}: TP={tp} (grün) · FP={fp} (rot) · miss: orange=partial, blau=zero')
ax.legend(handles=[mpatches.Patch(color='lime',label='TP pred'),mpatches.Patch(color='red',label='FP pred'),
                   mpatches.Patch(color='orange',label='GT partial (IoU<0.5)'),mpatches.Patch(color='blue',label='GT zero')],
          loc='upper right')
plt.tight_layout(); plt.show()

## 3. Zoom auf einzelne Kronen — Lokalisierung & Form im Detail

In [ ]:
# Zoom auf 6 zufällige 'partial'-Kronen (überlappt, aber IoU<0.5) — genau die strittigen
rng=np.random.default_rng(0)
partial_idx=[gi for gi,v in enumerate(best_iou_gt) if 0<v<IOU_MATCH]
pick=rng.choice(partial_idx, size=min(6,len(partial_idx)), replace=False) if partial_idx else []
HALF=90  # Pixel-Halbfenster
fig,axes=plt.subplots(2,3,figsize=(15,10)); axes=axes.ravel()
for k,gi in enumerate(pick):
    ax=axes[k]; g=gts[gi]; c,r=px(g); cy,cx=int(np.mean(r)),int(np.mean(c))
    ax.imshow(rgb); ax.set_xlim(cx-HALF,cx+HALF); ax.set_ylim(cy+HALF,cy-HALF); ax.axis('off')
    ax.plot(c,r,color='lime',lw=1.8)
    for p in preds:
        if p.bounds[0]>g.bounds[2] or p.bounds[2]<g.bounds[0] or p.bounds[1]>g.bounds[3] or p.bounds[3]<g.bounds[1]: continue
        pc,pr=px(p); ax.plot(pc,pr,color='red',lw=1.8)
    ax.set_title(f'GT #{gi}  best-IoU={best_iou_gt[gi]:.2f}', fontsize=10)
for k in range(len(pick),6): axes[k].axis('off')
fig.suptitle('Zoom: partial-Kronen (grün=GT, rot=Vorhersage)'); plt.tight_layout(); plt.show()

## Bewertung
- **Liegen die roten Vorhersagen auf den grünen/orangen GT-Kronen?** → Lokalisierung ok,
  der fehlende Recall bei IoU 0.5 ist überwiegend ein Form-/Größenproblem (partial), kein
  Platzierungsfehler.
- **Viele blaue GT ohne jede Vorhersage in der Nähe?** → echte Nicht-Erkennung (zero).
- Anderes Gebiet/Modell: oben `AREA` / `CONFIG` / `CHECKPOINT` ändern und neu durchlaufen.
